<a href="https://colab.research.google.com/github/sarbanibhadra/Forest-Biomass-Mapping/blob/main/RandomForest_Biomass_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Report: Above Ground Biomass (AGB) Mapping using Random Forest

## 1. Project Goal

The primary objective of this project is to develop and apply a Random Forest regression model to predict Above Ground Biomass (AGB) using a combination of remote sensing data from Earth Engine. The project includes data preparation, model training, evaluation, and analysis of feature importance and model uncertainties.

## 2. Data Sources and Preprocessing

The model utilizes various Earth Engine datasets as predictor variables, covering a specific study area within the Amazon basin (defined by `amazon_border`) and a specified date range. The following data sources were integrated:

*   **Sentinel-1 GRD (SAR Data)**: VV and VH polarizations were used, masked for noisy edges, and normalized. Ratios and differences (VV/VH, VV-VH) were also considered.
*   **Sentinel-2 MSI (Optical Data)**: Bands were used to calculate several vegetation and land-cover indices, including Normalized Difference Vegetation Index (NDVI), Enhanced Vegetation Index (EVI), Modified Normalized Difference Water Index (mNDWI), Normalized Difference Built-up Index (NDBI), and Bare Soil Index (BSI). Clouds and cirrus were masked using QA60 and Scene Classification Layer (SCL) bands.
*   **Landsat 8 Surface Reflectance**: Used to calculate a separate NDVI layer.
*   **Copernicus DEM (Digital Elevation Model)**: Provided elevation data.
*   **Derived Topographic Features**: Slope and aspect were calculated from the DEM.
*   **ESA WorldCover 2020**: Used to create a forest mask to focus predictions on forested areas.
*   **GEDI L4A (Global Ecosystem Dynamics Investigation)**: This dataset provided the observed AGB values, which served as the target variable for model training. GEDI data was filtered for quality, measurement errors (relative standard error < 50%), and slopes less than 30 degrees.

All predictor bands were stacked and reprojected to a common resolution (100m scale, EPSG:3857). Features were then normalized using min-max scaling based on statistics derived from the `amazon_border` region.

## 3. Model Training

### 3.1 Training Data Preparation

Training data was generated by combining the preprocessed feature stack with the filtered GEDI AGB mosaic. A stratified sampling approach was used across the `amazon_border` to create a `training` FeatureCollection, which was then split into a 70% training set and a 30% testing set using a random column.

### 3.2 Model Architecture

A Random Forest (RF) regression model (`ee.Classifier.smileRandomForest`) was employed to predict AGB. The model was configured with `numberOfTrees=200` and `bagFraction=0.99` and trained on the `training_set` using the prepared `predictors` and `agbd` as the target variable.

## 4. Model Evaluation

### 4.1 Initial Performance Metrics

Upon initial training, the model's performance was assessed using Root Mean Squared Error (RMSE) and R-squared (R2) on both the training and testing datasets:

*   **Training Dataset**:
    *   RMSE: 10.74 Mg/ha
    *   R2: 0.960
*   **Testing Dataset**:
    *   RMSE: 30.74 Mg/ha
    *   R2: 0.657

The significant difference between training and testing R2 values (high training R2, lower testing R2) indicates a degree of **overfitting**, meaning the model learned the training data very well but generalized less effectively to unseen data.

### 4.2 Observed vs. Predicted Scatter Plot

A scatter plot of observed AGB values against predicted AGB values on the testing set showed a general positive correlation, with points clustering around the 1:1 line. However, some scatter was visible, particularly at higher AGB values, confirming the model's generalization challenges. The legend clearly differentiates between the observed vs. predicted data points and the ideal 1:1 relationship.

### 4.3 Residual Analysis

Analysis of the residuals (observed - predicted values) on the testing set revealed:

*   **Mean Residual**: -1.00
*   **Standard Deviation of Residuals**: 30.73

This indicates a slight under-prediction bias on average, with residuals distributed around zero. The standard deviation provides a measure of the typical error magnitude.

## 5. Explainable AI: Feature Importance

To understand which input features contributed most to the AGB predictions, the feature importance was extracted from the trained Random Forest model. The bar plot visualization highlighted the following as the most important features:

1.  `DEM_mean`
2.  `aspect`
3.  `NDVI`
4.  `slope`

This suggests that topographic features and vegetation greenness indices are critical drivers for AGB estimation in this model.

## 6. Hyperparameter Tuning

To address the observed overfitting and improve testing performance, a systematic hyperparameter tuning process was initiated. Key hyperparameters explored included `numberOfTrees`, `bagFraction`, and `maxNodes`.

*   **`numberOfTrees`**: Varied as [200, 500, 800]
*   **`bagFraction`**: Varied as [0.6, 0.8, 0.99]
*   **`maxNodes`**: Varied as [10, 50, 100, 500]

The goal was to identify a combination that would yield the best R2 on the testing set, thereby improving generalization.

## 7. Data Imbalance Checks

### 7.1 Target Variable (`agbd`) Distribution

A histogram of the `agbd` values in the training set revealed a skewed distribution:

*   Minimum AGBD: 0.00 Mg/ha
*   Maximum AGBD: 534.26 Mg/ha
*   Mean AGBD: 116.14 Mg/ha
*   Median AGBD: 113.57 Mg/ha
*   Standard Deviation: 53.89 Mg/ha

The distribution is concentrated at lower to moderate AGB values, with fewer samples at very high AGB. This imbalance suggests the model might perform better in predicting common AGB ranges but could struggle with extreme values due to limited training data.

### 7.2 Predictor Feature Distributions

Histograms and descriptive statistics for each predictor feature (`VV_M`, `VH_M`, `S2_NDVI_median`, etc.) were also generated. These revealed varying degrees of skewness, which is typical for Earth observation data. However, Random Forest models are generally robust to skewed feature distributions, so this is not expected to be a major impediment to model performance.

## 8. AGB Map Prediction

Finally, the trained Random Forest model (`biomass_model`) was used to predict an AGB map (`agb_prediction_image`) over a new region and time period (July 2025). The predicted AGB image was visualized on the map with a custom color palette, showing AGB values ranging from 0 to 300 Mg/ha.

In [ ]:
# *****************************************************************
# Pipeline to generate AGB map based on random forest algorithm
# First Part --- Training the Model
# Date 29th October 2025
# *****************************************************************
import ee
import geemap
import folium
import time
import sys
import joblib
sys.setrecursionlimit(10000)  # Example: increase limit to 2000

ee.Authenticate()
ee.Initialize(project='project5324-448512')
Map = geemap.Map()
print("Start")
# '2021-06-01', '2023-12-31'
startDate = ee.Date.fromYMD(2021, 5, 1)
endDate = ee.Date.fromYMD(2021, 5, 16)

# tapajos_xingu = ee.Geometry.Polygon([
#   [[-55.5, -5.0], [-55.5, -2.0], [-51.5, -2.0], [-51.5, -5.0], [-55.5, -5.0]]
# ]); Covers mature forest zone between Tapajós and Xingu Rivers; Includes field research areas near Santarém and FLONA Tapajós

amazon_border = ee.Geometry.Polygon([[[-65.0, -5.0],
                                      [-65.0, 0.0],
                                      [-60.0, 0.0],
                                      [-60.0, -5.0],
                                      [-65.0, -5.0]]])
# amazon_border = ee.Geometry.Polygon([[[-80, -18], [-75, -18], [-75, -10], [-80, -10], [-80, -18]]])

# amazon_border = ee.FeatureCollection("projects/project5324-448512/assets/AmazonBasinLimits-master")

#  Featurestack preparation
# // Preparing sentinel-1
# // ****************************************************
# Define a masking function
def mask_edges(image):
    edge = image.lt(-30.0)  # Define an edge mask where values are less than -30
    masked_image = image.mask().And(edge.Not())  # Mask out edges
    return image.updateMask(masked_image)  # Apply the mask

sentinel1 = ee.ImageCollection('COPERNICUS/S1_GRD')\
            .filterBounds(amazon_border)\
            .filterDate(startDate, endDate) \
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
            .filter(ee.Filter.eq('instrumentMode', 'IW'))\
            .filter(ee.Filter.inList('orbitProperties_pass', ['ASCENDING', 'DESCENDING']))

# Optional: if not already in linear scale
sentinel1 = sentinel1.map(lambda img: img.expression('10**(b("VV") / 10)', {}).rename('VV')
                              .addBands(img.expression('10**(b("VH") / 10)', {}).rename('VH')))

# Select the VV and VH bands
sentinel1_vv = sentinel1.select('VV')
sentinel1_vh = sentinel1.select('VH')

# Apply the masking function to each image in the collection
sentinel1_vv_masked = sentinel1_vv.map(mask_edges).median().clip(amazon_border).unitScale(-30, 5).rename('S1_VV')

# Apply the masking function to each image in the collection
sentinel1_vh_masked = sentinel1_vh.map(mask_edges).median().clip(amazon_border).unitScale(-30, 5).rename('S1_VH')
sentinel1_masked = sentinel1_vv_masked.add(sentinel1_vh_masked).divide(2).rename('VV_VH')

# Compute VV/VH and VV - VH
vv_vh_ratio = sentinel1_vv_masked.divide(sentinel1_vh_masked).rename('VV_div_VH')
vv_minus_vh = sentinel1_vv_masked.subtract(sentinel1_vh_masked).rename('VV_minus_VH')

# load sentinel-2 data
sentinel2_raw = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
                .filterBounds(amazon_border) \
                .filterDate(startDate, endDate) \
                .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))

# Define the bitmasks
cloud_bit_mask = ee.Number(1 << 10)  # Cloud bit is in the 6th bit position
cirrus_bit_mask = ee.Number(1 << 11)  # Cirrus bit is in the 10th bit position

# Apply the mask using bitwise AND to check that both cloud and cirrus bits are 0
def mask_clouds(image):
    qa = image.select('QA60')  # Select the QA60 band that holds cloud and cirrus bit information
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask)

sentinel2 = sentinel2_raw.map(mask_clouds)

def mask_clouds_scl(image):
    scl = image.select('SCL')
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))  # 3: Shadow, 8: Cloud Medium Prob, 9: Cloud High Prob, 10: Cirrus
    return image.updateMask(mask)

sentinel2 = sentinel2.map(mask_clouds_scl)

# Calculate NDVI
ndvi = sentinel2.map(lambda image: image.normalizedDifference(['B8', 'B4'])).reduce(ee.Reducer.median()).rename('S2_NDVI')
# ndvi = ndvi.reproject(crs='EPSG:3857', scale=100)
print("ndvi-projection: ",ndvi.projection().getInfo())

# Calculate EVI
def calculate_evi(image):
    return image.expression(
        '2.5 * ((B8 - B4) / (B8 + 6 * B4 - 7.5 * B2 + 1))',
        {
            'B8': image.select('B8'),
            'B4': image.select('B4'),
            'B2': image.select('B2')
        }).rename('S2_EVI')

evi = sentinel2.map(calculate_evi).median()
# evi = evi.reproject(crs='EPSG:3857', scale=100)
print("evi-projection: ",evi.projection().getInfo())

mndwi = sentinel2.map(lambda image: image.normalizedDifference(['B3', 'B11'])).reduce(ee.Reducer.median()).rename('S2_MNDWI')
# mndwi = mndwi.reproject(crs='EPSG:3857', scale=100)
print("mndwi-projection: ",mndwi.projection().getInfo())

ndbi = sentinel2.map(lambda image: image.normalizedDifference(['B11', 'B8'])).reduce(ee.Reducer.median()).rename('S2_NDBI')
# ndbi = ndbi.reproject(crs='EPSG:3857', scale=100)
print("ndbi-projection: ",ndbi.projection().getInfo())

def calculate_bsi(image):
    return image.expression(
      '(( X + Y ) - (A + B)) /(( X + Y ) + (A + B)) ', {
        'X': image.select('B11'),
        'Y': image.select('B4'),
        'A': image.select('B8'),
        'B': image.select('B2'),
  }).rename('S2_BSI')

bsi = sentinel2.map(calculate_bsi).median()
# bsi = bsi.reproject(crs='EPSG:3857', scale=100)
print("bsi-projection: ",bsi.projection().getInfo())

# Load Landsat 8 Surface Reflectance data
landsat8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
              .filterBounds(amazon_border) \
              .filterDate(startDate, endDate) \
              .filter(ee.Filter.lt('CLOUD_COVER', 50))

# Calculate NDVI for Landsat 8
landsat_ndvi = landsat8.map(lambda image: image.normalizedDifference(['SR_B5', 'SR_B4']).rename('Landsat_NDVI')).median()
# landsat_ndvi = landsat_ndvi.reproject(crs='EPSG:3857', scale=100)
print("landsat_ndvi-projection: ",landsat_ndvi.projection().getInfo())

dem = ee.ImageCollection('COPERNICUS/DEM/GLO30') \
            .filterBounds(amazon_border) \
            .select('DEM') \
            .reduce(ee.Reducer.mean())\
            .clip(amazon_border)\
            .reproject('EPSG:3857', None, 500)\
            .rename('DEM')
# dem = dem.reproject(crs='EPSG:3857', scale=100)

# Calculate slope in degrees
slope = ee.Terrain.slope(dem).rename('Slope')

# Calculate aspect in degrees
aspect = ee.Terrain.aspect(dem).rename('Aspect')

# Load ESA WorldCover 2020 dataset
esa_worldcover = ee.ImageCollection('ESA/WorldCover/v100').filterBounds(amazon_border).first();
# Mask non-forest pixels (keep only tree cover)
forest_mask = esa_worldcover.eq(10)  # Tree cover class = 10
forest = esa_worldcover.updateMask(forest_mask).clip(amazon_border)
# forest = forest.reproject(crs='EPSG:3857', scale=100)
print("Type of forest_only", type(forest))
print("forest-projection: ",forest.projection().getInfo())

# Load the JRC Global Surface Water dataset
water_dataset = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')

# Use the "occurrence" band to represent water presence (values > 50 indicate permanent water presence)
river_band = water_dataset.select('occurrence').gt(50).rename('water_presence').clip(amazon_border)

# Optionally convert river_band to float
river_band = river_band.toFloat()

# Latitude and longitude bands
lon = ee.Image.pixelLonLat().select('longitude').rename('lon')
lat = ee.Image.pixelLonLat().select('latitude').rename('lat')

# Combine all bands into a single feature stack
feature_stack = sentinel1_vv_masked.addBands(sentinel1_vh_masked) \
    .addBands(ndvi) \
    .addBands(mndwi) \
    .addBands(ndbi) \
    .addBands(bsi) \
    .addBands(evi) \
    .addBands(landsat_ndvi) \
    .addBands(dem) \
    .addBands(slope) \
    .addBands(aspect)

# Reproject feature stack for model training
feature_stack = feature_stack.reproject(crs='EPSG:3857', scale=100)
feature_stack = feature_stack.clip(amazon_border)
bands = ['S1_VV', 'S1_VH', 'S2_NDVI', 'S2_MNDWI', 'S2_NDBI', 'S2_BSI', 'S2_EVI', 'Landsat_NDVI', 'DEM', 'Slope', 'Aspect']

band_stats = feature_stack.select(bands).reduceRegion(ee.Reducer.minMax().combine(ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs= True), sharedInputs=True),
                                                      geometry=amazon_border,
                                                      scale=100,
                                                      maxPixels=1e13)

# print('Stats:', band_stats.getInfo())
def normalize_image(image, bands, stats):
    i = 0
    for band in bands:
        print(band)
        min_val = ee.Number(stats.get(f'{band}_min'))
        max_val = ee.Number(stats.get(f'{band}_max'))
        norm = (
            image.select(band)
            .subtract(min_val)
            .divide(max_val.subtract(min_val))
            .rename(band)
        )
        if i>0:
            print(i)
            norm_bands = norm_bands.addBands(ee.Image(norm))
        else:
            norm_bands = ee.Image(norm)
        i= i+1
        print(type(norm))
    return ee.Image(norm_bands)

normFeatureStack = normalize_image(feature_stack, bands, band_stats)

print("Type of featute_stack", type(feature_stack))

print("Type of normFeatureStack", type(normFeatureStack))
# Map.addLayer(normStack, {}, 'Normalized Inputs');

# Stratified sampling
# // Preparing GEDI L4A Mosaic
# // ****************************************************
gedi = ee.ImageCollection('LARSE/GEDI/GEDI04_A_002_MONTHLY')
print(gedi.size().getInfo())
 # Function to select highest quality GEDI data
def qualityMask(image):
    return image.updateMask(image.select('l4_quality_flag').eq(1)).updateMask(image.select('degrade_flag').eq(0))

# Function to mask unreliable GEDI measurements
# with a relative standard error > 50%
# agbd_se / agbd > 0.5
def errorMask(image):
    relative_se = image.select('agbd_se').divide(image.select('agbd'))
    return image.updateMask(relative_se.lte(0.5))

# Function to mask GEDI measurements on slopes > 30%
def slopeMask(image) :
    return image.updateMask(slope.lt(30))

def nullmask(image):
    return image.select('agbd').neq('null')

def outliermask(image):
    return image.select('agbd').gt(0)
# Remove or flag outlier points in the AGB (e.g., values > 400 Mg/ha in sparse vegetation).
# gedi = gedi.filter(ee.Filter.lt('agbd', 400))
gediFiltered = gedi.filter(ee.Filter.date(startDate, endDate)).filter(ee.Filter.bounds(amazon_border));
gediProcessed = gediFiltered.map(qualityMask).map(errorMask).map(slopeMask);

gediProjection = ee.Image(gediFiltered.first()).select('agbd').projection();
gediMosaic = gediProcessed.mosaic().select('agbd').setDefaultProjection(gediProjection);
print(gediProcessed.size().getInfo())
# // ****************************************************

stacked = normFeatureStack.addBands(gediMosaic)
stacked = stacked.resample('bilinear')

gridScale = 100;
gridProjection = ee.Projection('EPSG:3857').atScale(gridScale);

# Aggregate pixels with 'mean' statistics
stackedResampled = stacked.reduceResolution(ee.Reducer.mean()).reproject(gridProjection)
stackedResampled = stackedResampled.updateMask(stackedResampled.mask().gt(0))

# Combine all bands into a single feature stack
predictors = ['S1_VV', 'S1_VH', 'S2_NDVI', 'S2_MNDWI', 'S2_NDBI', 'S2_BSI', 'S2_EVI', 'Landsat_NDVI', 'DEM', 'Slope', 'Aspect']
predicted = 'agbd'
print('predictors:', predictors)
print('predicted:', predicted)

predictorImage = stackedResampled.select(predictors)
predictedImage = stackedResampled.select([predicted])

classMask = predictedImage.mask().toInt().rename('agb');

numSamples = 3000

training = stackedResampled.addBands(classMask).stratifiedSample(
    numPoints= numSamples,
    classBand= 'agb',
    region= amazon_border,
    scale= gridScale,
    classValues= [0, 1],
    classPoints= [0, numSamples],
    dropNulls= True,
    tileScale= 4,
)
# print(training.size().getInfo())
# Split data into training and testing sets
col = training.randomColumn()
training_set = col.filter(ee.Filter.lt('random', 0.7))
# print(training_set.size().getInfo())
testing_set  = col.filter(ee.Filter.gte('random', 0.3))
# print(testing_set.size().getInfo())
# print('Number of Features Extracted', training.size().getInfo());
# print('Sample Training Feature', training.first().getInfo());
# ee.Classifier.smileRandomForest(numberOfTrees, variablesPerSplit, minLeafPopulation, bagFraction, maxNodes, seed)

numberOfTrees = 900
maxNodes = 2
minLeafPopulation = 2
bagFraction = 0.9
variablesPerSplit = int(len(predictors) ** 0.5)  # sqrt rule

biomass_model = ee.Classifier.smileRandomForest(numberOfTrees=200, bagFraction=0.99 ).setOutputMode('REGRESSION').train(
    features= training_set,
    classProperty= predicted,
    inputProperties= predictors
  )

predicted_map_traning = training_set.classify(classifier = biomass_model, outputName= 'agbd_predicted' )
predicted_map = testing_set.classify(classifier = biomass_model, outputName= 'agbd_predicted' )
# print(predicted_map.first().getInfo())

# def calculateRmse(input):
#     observed = ee.Array(input.aggregate_array('agbd'))
#     predicted = ee.Array(input.aggregate_array('agbd_predicted'))
#     rmse = observed.subtract(predicted).pow(2).reduce('mean', [0]).sqrt().get([0])
#     return rmse

# rmse = calculateRmse(predicted_map);
# print('RMSE', rmse.getInfo())
# print("Year: 2021")


def calculateRmse(input):
    observed = ee.Array(input.aggregate_array('agbd'))
    predicted = ee.Array(input.aggregate_array('agbd_predicted'))
    observed_mean = observed.reduce(ee.Reducer.mean(), [0])
    rmse = observed.subtract(predicted).pow(2).reduce('mean', [0]).sqrt().get([0])
    return rmse

def calculateR2(input):
    observed = ee.Array(input.aggregate_array('agbd'))
    predicted = ee.Array(input.aggregate_array('agbd_predicted'))
    observed_mean = observed.reduce(ee.Reducer.mean(), [0]).get([0])
    SSres = observed.subtract(predicted).pow(2).reduce('sum', [0]).get([0])
    SStotal = observed.subtract(observed_mean).pow(2).reduce('sum', [0]).get([0])
    R2 = ee.Number(1).subtract(SSres.divide(SStotal))
    return R2

rmse_traning = calculateRmse(predicted_map_traning)
r2_traning= calculateR2(predicted_map_traning)
print('RMSE on training dataset', rmse_traning.getInfo())
print('R2 on training dataset', r2_traning.getInfo())
rmse_testing = calculateRmse(predicted_map)
r2_testing= calculateR2(predicted_map)
print('RMSE on testing dataset', rmse_testing.getInfo())
print('R2 on testing dataset', r2_testing.getInfo())
print("Year: 2021")
joblib.dump(biomass_model, "biomass_model_RMSEV2_NORM.pkl")
print("Training completed!")

predictedImageDemo = stackedResampled.classify(classifier= biomass_model, outputName= 'agbd' )
# Visualization
rgbVis = { min: 0.0, max: 0.3 }
gediVis = { min: 0, max: 300, 'palette': ['#edf8fb','#b2e2e2','#66c2a4','#2ca25f','#006d2c'], 'bands': ['agbd']}

Map.addLayer(vv_vh_ratio,
  {'min': 0.5, 'max': 3.0},
  'sentinel1'
);
Map.addLayer(sentinel2_raw.median().clip(amazon_border),
  {'min': 0, 'max': 1000},
  'sentinel2'
);
Map.addLayer(forest,
  {'min': 0, 'max': 1},
  'forest'
);
Map.addLayer(ndvi.clip(amazon_border),
  {'min': 0, 'max': 1, 'palette': ['blue', 'teal', 'green']},
  'ndvi'
);
Map.addLayer(evi.clip(amazon_border),
  {'min': -25, 'max': 50},
  'evi'
);
Map.addLayer(landsat_ndvi.clip(amazon_border),
  {'min': -25, 'max': 50},
  'landsat_ndvi'
);
Map.addLayer(dem,
  {'min': 0, 'max': 1000},
  'dem'
);
Map.addLayer(slope,
  {'min': 0, 'max': 50},
  'slope'
);
Map.addLayer(aspect.clip(amazon_border),
  {'min': 0, 'max': 360},
  'aspect'
);
Map.addLayer(feature_stack.clip(amazon_border),
  {'min': -25, 'max': 5},
  'feature_stack'
);
Map.addLayer(
  stackedResampled, rgbVis, 'Sentinel-2 (Resampled)'
);
Map.addLayer(
  stackedResampled, gediVis, 'GEDI L4A (Resampled)'
);
Map.addLayer(feature_stack.clip(amazon_border),
  {'min': 0, 'max': 1},
  'feature_stack'
);
Map.addLayer(gediFiltered.mosaic().clip(amazon_border),
  gediVis, 'GEDI L4A (Raw)'
);
Map.addLayer(gediMosaic.clip(amazon_border),
  gediVis, 'GEDI L4A (Filtered)'
);
Projection = ee.Projection('EPSG:3857').atScale(5000);
agb_vis_5000m = predictedImageDemo.reproject(Projection)
Map.addLayer(agb_vis_5000m, {
    'min': 0, 'max': 300,
    'palette': ['white', 'blue', 'FFFF00', 'ADFF2F', '7CFC00', '32CD32', '228B22', '008000', '006400', '004d00'], 'bands': ['agbd']}, 'agb_prediction_image_5km'
);
Map.addLayer(amazon_border, {}, 'Amazon Border')
Map.add_layer_control()
Map

Start
ndvi-projection:  {'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}
evi-projection:  {'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}
mndwi-projection:  {'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}
ndbi-projection:  {'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}
bsi-projection:  {'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}
landsat_ndvi-projection:  {'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}
Type of forest_only <class 'ee.image.Image'>
forest-projection:  {'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [8.333333333333333e-05, 0, -180, 0, -8.333333333333333e-05, 84]}
S1_VV
<class 'ee.image.Image'>
S1_VH
1
<class 'ee.image.Image'>
S2_NDVI
2
<class 'ee.image.Image'>
S2_MNDWI
3
<class 'ee.image.Image'>
S2_NDBI
4
<class 'ee.image.Image'>
S2_BSI
5
<class 'ee.image.Image'>
S2_EVI
6
<class 'ee.image.Image'>

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [ ]:
Map.addLayer(agb_vis_5000m, {
    'min': 0, 'max': 300,
    'palette': ['white', 'blue', 'FFFF00', 'ADFF2F', '7CFC00', '32CD32', '228B22', '008000', '006400', '004d00'], 'bands': ['agbd']}, 'agb_prediction_image_5km'
);
Map.addLayer(amazon_border, {}, 'Amazon Border')
Map.add_layer_control()
Map


predictors: ['VV_M', 'VH_M', 'S2_NDVI_median', 'mndwi_median', 'ndbi_median', 'bsi', 'S2_EVI', 'NDVI', 'DEM_mean', 'slope', 'aspect']
predicted: agbd
RMSE on traning dataset 10.74386771688026
R2 on traning dataset 0.9602522744584729
RMSE on testing dataset 30.748256758832675
R2 on traning dataset 0.6568858725930145
Year: 2021
Training completed!

In [ ]:
biomass_model = ee.Classifier.smileRandomForest(numberOfTrees=300, bagFraction=0.9 ).setOutputMode('REGRESSION').train(
    features= training_set,
    classProperty= predicted,
    inputProperties= predictors
  )

predicted_map_training = training_set.classify(classifier = biomass_model, outputName= 'agbd_predicted' )
predicted_map = testing_set.classify(classifier = biomass_model, outputName= 'agbd_predicted' )
rmse_traning = calculateRmse(predicted_map_traning)
r2_traning= calculateR2(predicted_map_traning)
print('RMSE on training dataset', rmse_traning.getInfo())
print('R2 on training dataset', r2_traning.getInfo())
rmse_testing = calculateRmse(predicted_map)
r2_testing= calculateR2(predicted_map)
print('RMSE on testing dataset', rmse_testing.getInfo())
print('R2 on testing dataset', r2_testing.getInfo())
print("Year: 2021")

In [ ]:
biomass_model3 = ee.Classifier.smileRandomForest(numberOfTrees=300, bagFraction=0.9 ).setOutputMode('REGRESSION').train(
    features= training_set,
    classProperty= predicted,
    inputProperties= predictors
  )

predicted_map_training = training_set.classify(classifier = biomass_model3, outputName= 'agbd_predicted' )
predicted_map_testing = testing_set.classify(classifier = biomass_model3, outputName= 'agbd_predicted' )
rmse_traning = calculateRmse(predicted_map_training)
r2_traning= calculateR2(predicted_map_training)
print('RMSE on training dataset', rmse_traning.getInfo())
print('R2 on training dataset', r2_traning.getInfo())
rmse_testing = calculateRmse(predicted_map_testing)
r2_testing= calculateR2(predicted_map_testing)
print('RMSE on testing dataset', rmse_testing.getInfo())
print('R2 on testing dataset', r2_testing.getInfo())
print("Year: 2021")

In [ ]:
numberOfTrees = 900
maxNodes = 2
minLeafPopulation = 2
bagFraction = 0.9
variablesPerSplit = int(len(predictors) ** 0.5)  # sqrt rule
biomass_model4 = ee.Classifier.smileRandomForest(numberOfTrees=250, maxNodes = 500, bagFraction=0.5 ).setOutputMode('REGRESSION').train(
    features= training_set,
    classProperty= predicted,
    inputProperties= predictors
  )

predicted_map_training = training_set.classify(classifier = biomass_model4, outputName= 'agbd_predicted' )
predicted_map_testing = testing_set.classify(classifier = biomass_model4, outputName= 'agbd_predicted' )
rmse_traning = calculateRmse(predicted_map_training)
r2_traning= calculateR2(predicted_map_training)
print('RMSE on training dataset', rmse_traning.getInfo())
print('R2 on training dataset', r2_traning.getInfo())
rmse_testing = calculateRmse(predicted_map_testing)
r2_testing= calculateR2(predicted_map_testing)
print('RMSE on testing dataset', rmse_testing.getInfo())
print('R2 on testing dataset', r2_testing.getInfo())
print("Year: 2021")

In [ ]:
numberOfTrees = 900
maxNodes = 2
minLeafPopulation = 2
bagFraction = 0.9
variablesPerSplit = int(len(predictors) ** 0.5)  # sqrt rule
biomass_model4 = ee.Classifier.smileRandomForest(numberOfTrees=1000, bagFraction=0.9 ).setOutputMode('REGRESSION').train(
    features= training_set,
    classProperty= predicted,
    inputProperties= predictors
  )

predicted_map_training = training_set.classify(classifier = biomass_model4, outputName= 'agbd_predicted' )
predicted_map_testing = testing_set.classify(classifier = biomass_model4, outputName= 'agbd_predicted' )
rmse_traning = calculateRmse(predicted_map_training)
r2_traning= calculateR2(predicted_map_training)
print('RMSE on training dataset', rmse_traning.getInfo())
print('R2 on training dataset', r2_traning.getInfo())
rmse_testing = calculateRmse(predicted_map_testing)
r2_testing= calculateR2(predicted_map_testing)
print('RMSE on testing dataset', rmse_testing.getInfo())
print('R2 on testing dataset', r2_testing.getInfo())
print("Year: 2021")

In [ ]:
biomass_model5 = ee.Classifier.smileRandomForest(numberOfTrees=250, bagFraction=0.99 ).setOutputMode('REGRESSION').train(
    features= training_set,
    classProperty= predicted,
    inputProperties= predictors
  )

predicted_map_training = training_set.classify(classifier = biomass_model5, outputName= 'agbd_predicted' )
predicted_map_testing = testing_set.classify(classifier = biomass_model5, outputName= 'agbd_predicted' )
rmse_traning = calculateRmse(predicted_map_training)
r2_traning= calculateR2(predicted_map_training)
print('RMSE on training dataset', rmse_traning.getInfo())
print('R2 on training dataset', r2_traning.getInfo())
rmse_testing = calculateRmse(predicted_map_testing)
r2_testing= calculateR2(predicted_map_testing)
print('RMSE on testing dataset', rmse_testing.getInfo())
print('R2 on testing dataset', r2_testing.getInfo())
print("Year: 2021")

In [ ]:
biomass_model6 = ee.Classifier.smileRandomForest(numberOfTrees=250,  bagFraction=0.99 ).setOutputMode('REGRESSION').train(
    features= training_set,
    classProperty= predicted,
    inputProperties= predictors
  )

predicted_map_training = training_set.classify(classifier = biomass_model6, outputName= 'agbd_predicted' )
predicted_map_testing = testing_set.classify(classifier = biomass_model6, outputName= 'agbd_predicted' )
rmse_traning = calculateRmse(predicted_map_training)
r2_traning= calculateR2(predicted_map_training)
print('RMSE on training dataset', rmse_traning.getInfo())
print('R2 on training dataset', r2_traning.getInfo())
rmse_testing = calculateRmse(predicted_map_testing)
r2_testing= calculateR2(predicted_map_testing)
print('RMSE on testing dataset', rmse_testing.getInfo())
print('R2 on testing dataset', r2_testing.getInfo())
print("Year: 2021")

## Explainable AI: Feature Importance

To understand which features are most important for the model's predictions, we can extract the feature importance from the trained Random Forest classifier.

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Get the feature importance from the trained model
# The biomass_model object from cell 8f2be092-c9be-4460-8d9a-34025d447d24 is used here.
# If you ran other models (e.g., biomass_model6), you might want to change this.
explanation = biomass_model.explain().getInfo()

# Extract the nested feature importance dictionary
feature_importance_data = explanation.get('importance', {})

# Convert to a pandas DataFrame for better readability
feature_importance_df = pd.DataFrame({
    'Feature': list(feature_importance_data.keys()),
    'Importance': list(feature_importance_data.values())
})

# Sort by importance in descending order
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Scale the 'Importance' column to a 0-1 range
scaler = MinMaxScaler()
feature_importance_df['Scaled Importance'] = scaler.fit_transform(feature_importance_df[['Importance']])

print("Feature Importance for AGB Prediction (Scaled):")
display(feature_importance_df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 7))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df, palette='viridis')
plt.title('Feature Importance for AGB Prediction')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## Hyperparameter Tuning for Random Forest

To improve the model's performance on the testing dataset (reduce overfitting and increase R2), we can systematically tune the hyperparameters of the `ee.Classifier.smileRandomForest` model. Key hyperparameters to consider include:

*   **`numberOfTrees`**: The number of decision trees in the forest. Increasing this can reduce variance but also increases computation time. Too many trees might not significantly improve performance beyond a certain point.
*   **`bagFraction`**: The fraction of input samples to draw (without replacement) for each tree. Lower values (e.g., 0.6 to 0.8) can introduce more diversity among trees, potentially reducing overfitting.
*   **`minLeafPopulation`**: The minimum number of data points required to be in a leaf node. Increasing this can prevent the model from learning overly specific patterns from small subsets of data.
*   **`maxNodes`**: The maximum number of terminal nodes (leaves) in each tree. Limiting this can reduce the depth and complexity of individual trees, which helps in preventing overfitting.
*   **`variablesPerSplit`**: The number of features to consider when looking for the best split. The square root of the total number of features is a common heuristic for regression tasks, but experimenting with other values can be beneficial.

Given your current high training R2 and lower testing R2, we will focus on adjusting `bagFraction` and `maxNodes` to reduce overfitting. We'll iterate through a few combinations to see how they impact the RMSE and R2 on both training and testing datasets.

In [ ]:
print('Starting hyperparameter tuning...')

best_test_r2 = -float('inf')
best_params = {}

# Define a range of hyperparameters to explore
tree_counts = [200, 500, 800, 1200] # Expanded: Up to 1200 trees
bag_fractions = [0.6, 0.7, 0.8, 0.9, 0.99] # More granular steps
max_node_counts = [10, 50, 100, 200, 500, None] # Expanded, including no limit (None)
min_leaf_populations = [1, 2, 5, 10] # New parameter: Min samples in a leaf

# Note: 'variablesPerSplit' can also be tuned, but for simplicity, we'll keep the sqrt rule for now.
# variables_per_split_options = [int(len(predictors) ** 0.5), int(len(predictors) * 0.7), len(predictors)]

# Loop through combinations of hyperparameters
for num_trees in tree_counts:
    for bag_frac in bag_fractions:
        for max_nodes_val in max_node_counts:
            for min_leaf_pop_val in min_leaf_populations:
                print(f"\nTraining with: numberOfTrees={num_trees}, bagFraction={bag_frac}, maxNodes={max_nodes_val}, minLeafPopulation={min_leaf_pop_val}")

                # Train the Random Forest classifier with current hyperparameters
                current_biomass_model = ee.Classifier.smileRandomForest(
                    numberOfTrees=num_trees,
                    bagFraction=bag_frac,
                    maxNodes=max_nodes_val,
                    minLeafPopulation=min_leaf_pop_val
                ).setOutputMode('REGRESSION').train(
                    features=training_set,
                    classProperty=predicted,
                    inputProperties=predictors
                )

                # Classify training and testing sets
                predicted_map_training = training_set.classify(classifier=current_biomass_model, outputName='agbd_predicted')
                predicted_map_testing = testing_set.classify(classifier=current_biomass_model, outputName='agbd_predicted')

                # Calculate RMSE and R2 for training set
                rmse_training = calculateRmse(predicted_map_training)
                r2_training = calculateR2(predicted_map_training)
                print('  RMSE on training dataset', rmse_training.getInfo())
                print('  R2 on training dataset', r2_training.getInfo())

                # Calculate RMSE and R2 for testing set
                rmse_testing = calculateRmse(predicted_map_testing)
                r2_testing = calculateR2(predicted_map_testing)
                print('  RMSE on testing dataset', rmse_testing.getInfo())
                print('  R2 on testing dataset', r2_testing.getInfo())

                # Keep track of the best model based on testing R2
                if r2_testing.getInfo() > best_test_r2:
                    best_test_r2 = r2_testing.getInfo()
                    best_params = {
                        'numberOfTrees': num_trees,
                        'bagFraction': bag_frac,
                        'maxNodes': max_nodes_val,
                        'minLeafPopulation': min_leaf_pop_val
                    }
                    # You might want to save the best model here if needed
                    # joblib.dump(current_biomass_model, "best_biomass_model.pkl")

print("\nHyperparameter tuning complete!")
print(f"Best Testing R2: {best_test_r2:.4f} with parameters: {best_params}")

# You can then retrain your final model with the best_params found
# or directly use the 'current_biomass_model' if it was the last best one
# biomass_model = ee.Classifier.smileRandomForest(
#     numberOfTrees=best_params['numberOfTrees'],
#     bagFraction=best_params['bagFraction'],
#     maxNodes=best_params['maxNodes'],
#     minLeafPopulation=best_params['minLeafPopulation']
# ).setOutputMode('REGRESSION').train(
#     features=training_set,
#     classProperty=predicted,
#     inputProperties=predictors
# )

## Uncertainty Quantification: Residual Analysis

To quantify the uncertainty of the model's predictions, we can analyze the residuals, which are the differences between the observed and predicted Above Ground Biomass (AGB) values in the testing set. By examining their distribution, we can understand the typical error magnitude and potential biases.

## Explainable AI: SHAP (SHapley Additive exPlanations)

SHAP (SHapley Additive exPlanations) is a game theory approach to explain the output of any machine learning model. It connects optimal credit allocation with local explanations by using shapley values. This analysis will help us understand how each feature influences the model's predictions for individual instances.

In [ ]:
pip install shap

In [ ]:
import shap
import numpy as np

# Due to the nature of Earth Engine and potential limitations in directly
# exporting large datasets for client-side SHAP analysis, we'll focus on
# a subsample of the testing data if 'predicted_map' is very large.

# Ensure we have the observed and predicted values from the testing set
# These were already extracted in the previous cell for residual analysis.
# If not available, you would need to re-run the extraction here.

# For SHAP, we need the feature values (predictors) and the model's prediction function.
# Earth Engine's Classifier.classify method doesn't directly expose feature contributions
# in a way that directly integrates with the 'shap' library for tree-based models.
# However, we can approximate by creating a local 'predict' function that mimics
# the Earth Engine classification on a small, representative sample of features.

# It's important to note that direct SHAP analysis on a large Earth Engine
# dataset is complex due to data transfer limitations. This example focuses on
# demonstrating the SHAP concept by sampling features and using a placeholder
# for the prediction function.

# First, let's get the features (predictors) from a subset of the testing data.
# We will extract 'predictors' and 'agbd' for a small sample to create a local dataset.

# Define the number of samples for local SHAP analysis
shap_sample_size = 100 # Adjust this based on computational limits

# Create a feature collection with predictors and observed AGB for SHAP sampling
shap_features_collection = predicted_map.select(predictors + ['agbd'])

# Sample a small number of points for local SHAP analysis
local_shap_sample = shap_features_collection.limit(shap_sample_size).getInfo()['features']

# Extract features and target from the sampled data
feature_data = []
# Assuming 'agbd' is the target, we don't need it for the explainer's input.
# We only need the predictor values.
for f in local_shap_sample:
    props = f['properties']
    feature_data.append([props[p] for p in predictors])

# Convert to a numpy array for SHAP
X_shap = np.array(feature_data)

# Create a dummy model prediction function for demonstration.
# In a real scenario, you would need to export the trained EE Classifier
# to a local format (e.g., scikit-learn if it's a compatible model) or
# implement a way to get predictions from EE for specific local inputs.

# Placeholder for a local prediction function.
# This function would ideally call the Earth Engine model to classify local data.
# For now, it will return a simple mock prediction.
# In a real scenario, you would need to find a way to make local predictions from your EE model.
# One common approach for tree models is to export the model parameters and recreate it locally,
# or to use a small sample and classify it within EE, then transfer predictions.

def local_predict_function(X):
    # This is a placeholder. In a real application, you'd need to convert X back
    # to ee.FeatureCollection and use biomass_model.classify().
    # For this example, we'll just return an average of the observed agbd from the sample
    # or a dummy value. This will *not* reflect the actual EE model's behavior precisely.
    print("Warning: Using a placeholder local_predict_function. Actual EE model predictions cannot be directly mimicked here without more complex data export/re-creation.")
    return np.mean([f['properties']['agbd'] for f in local_shap_sample]) * np.ones(X.shape[0])

# Use the KernelExplainer for model-agnostic explanations if direct tree explainer is not feasible
# For tree models like RandomForest, shap.TreeExplainer is more efficient if the model can be directly accessed.
# If the Earth Engine model cannot be directly passed to TreeExplainer, KernelExplainer is a general alternative.

# Since biomass_model is an Earth Engine object and not directly usable by shap.TreeExplainer,
# we'll use KernelExplainer, which requires a background dataset and a prediction function.

# For the background dataset, we can use a small sample of our training data.
# Let's take a small subset of X_shap as background.
background = shap.utils.sample(X_shap, 10)

# Create the SHAP explainer
explainer = shap.KernelExplainer(local_predict_function, background)

# Calculate SHAP values for a subset of the test data (e.g., first 50 instances)
shap_values = explainer.shap_values(X_shap[:50])

# Visualize the SHAP values for the first prediction
shap.initjs()
shap.force_plot(explainer.expected_value, shap_values[0], X_shap[0], feature_names=predictors)

# Summary plot of feature importance based on SHAP values
shap.summary_plot(shap_values, X_shap[:50], feature_names=predictors)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# The 'predicted_map' contains both 'agbd' (observed) and 'agbd_predicted' (predicted) for the testing set.
# Extract these values for analysis.
# Note: This operation can be computationally intensive for large datasets.
# For demonstration, we'll try to sample if direct aggregation is too slow.

# Aggregate observed and predicted values from the testing set
observed_agbd = ee.Array(predicted_map.aggregate_array('agbd'))
predicted_agbd = ee.Array(predicted_map.aggregate_array('agbd_predicted'))


# Convert EE Arrays to client-side lists/numpy arrays for local processing and plotting
try:
    observed_values = observed_agbd.getInfo()
    predicted_values = predicted_agbd.getInfo()

    # Ensure they are lists of numbers, not nested lists if EE returns them that way
    observed_values = [item for sublist in observed_values for item in sublist] if any(isinstance(i, list) for i in observed_values) else observed_values
    predicted_values = [item for sublist in predicted_values for item in sublist] if any(isinstance(i, list) for i in predicted_values) else predicted_values

    observed_values = np.array(observed_values)
    predicted_values = np.array(predicted_values)
    print(observed_values.min(), observed_values.max() )

    # Calculate residuals
    residuals = observed_values - predicted_values

    # Calculate descriptive statistics of residuals
    mean_residual = np.mean(residuals)
    std_residual = np.std(residuals)

    print(f"Mean Residual: {mean_residual:.2f}")
    print(f"Standard Deviation of Residuals: {std_residual:.2f}")

    # Plot histogram of residuals
    plt.figure(figsize=(10, 6))
    plt.hist(residuals, bins=50, edgecolor='black')
    plt.title('Distribution of Prediction Residuals (Observed - Predicted)')
    plt.xlabel('Residual (AGB units)')
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.show()

    # Plot observed vs. predicted values
    plt.figure(figsize=(10, 6))
    plt.scatter(observed_values, predicted_values, alpha=0.3, label='Observed vs. Predicted')
    plt.plot([min(observed_values), max(observed_values)], [min(observed_values), max(observed_values)], 'r--', label='1:1 line')
    plt.title('Observed vs. Predicted AGB on Testing Set')
    plt.xlabel('Observed AGB')
    plt.ylabel('Predicted AGB')
    plt.grid(True)
    plt.legend()
    plt.show()

except Exception as e:
    print(f"Could not retrieve or process data for residual analysis. Error: {e}")
    print("This might happen if the aggregated array is too large for client-side processing.")
    print("Consider reducing the 'numSamples' in the training cell or performing analysis directly in Earth Engine if possible.")

## Class Imbalance Check

To assess potential class imbalance, we will inspect the distribution of the `agbd` (Above Ground Biomass) values in the training dataset. Imbalance can affect model training, particularly in regression tasks where certain ranges of the target variable might be underrepresented.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract 'agbd' values from the training set
observed_agbd_training = ee.Array(training_set.aggregate_array('agbd'))

try:
    # Convert EE Array to a client-side numpy array for plotting
    agbd_values = observed_agbd_training.getInfo()
    agbd_values = [item for sublist in agbd_values for item in sublist] if any(isinstance(i, list) for i in agbd_values) else agbd_values
    agbd_values = np.array(agbd_values)

    # Plot histogram of AGBD values in the training set
    plt.figure(figsize=(10, 6))
    plt.hist(agbd_values, bins=50, edgecolor='black')
    plt.title('Distribution of AGBD Values in Training Set')
    plt.xlabel('AGBD (Mg/ha)')
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.show()

    print(f"Minimum AGBD in training data: {np.min(agbd_values):.2f}")
    print(f"Maximum AGBD in training data: {np.max(agbd_values):.2f}")
    print(f"Mean AGBD in training data: {np.mean(agbd_values):.2f}")
    print(f"Median AGBD in training data: {np.median(agbd_values):.2f}")
    print(f"Standard Deviation of AGBD in training data: {np.std(agbd_values):.2f}")

except Exception as e:
    print(f"Could not retrieve or process AGBD training data for imbalance check. Error: {e}")
    print("This might happen if the aggregated array is too large for client-side processing.")

The distribution of Above Ground Biomass (AGBD) values in your training data has been visualized and summarized.

Here are the key statistics:

Minimum AGBD: 0.00 Mg/ha
Maximum AGBD: 534.26 Mg/ha
Mean AGBD: 116.14 Mg/ha
Median AGBD: 113.57 Mg/ha
Standard Deviation of AGBD: 53.89 Mg/ha
The histogram below visually represents this distribution, showing the frequency of different AGBD values. This information can help assess if there are certain AGBD ranges that are underrepresented, which might indicate class imbalance for regression tasks. The histogram you are referring to shows the Distribution of AGBD Values in the Training Set.

X-axis (AGBD (Mg/ha)): This axis represents the range of Above Ground Biomass Density values, measured in Megagrams per hectare (Mg/ha).
Y-axis (Frequency): This axis indicates how many data points (samples) from your training set fall within each specific range of AGBD values (each 'bin' of the histogram).
Essentially, it visualizes how frequently different AGBD values appear in your training data. A histogram helps you quickly understand:

The central tendency (where most values are concentrated).
The spread of the data (how wide the range of values is).
The shape of the distribution (e.g., if it's skewed, bimodal, or normally distributed).
Any outliers or gaps in the data.
In the context of checking for class imbalance in a regression task, a histogram like this helps you see if certain ranges of AGBD are over- or under-represented in your training data. For example, if there are very few samples for high AGBD values, your model might struggle to accurately predict AGBD in that range.
Based on the statistics and typical AGBD distributions, your training dataset does show some degree of imbalance in the target variable, agbd.

Here's why:

Wide Range: The AGBD values range from a minimum of 0.00 Mg/ha to a maximum of 534.26 Mg/ha. This is a very broad range.
Mean vs. Median: The mean (116.14 Mg/ha) and median (113.57 Mg/ha) are relatively close. This might suggest a somewhat symmetrical distribution if we only consider the central tendency, but it doesn't tell the whole story about the tails of the distribution.
Standard Deviation: A standard deviation of 53.89 Mg/ha indicates a significant spread around the mean. When combined with the wide range and a mean that is much closer to the minimum than the maximum, it suggests that while there are high AGBD values, they are likely less frequent than values around the mean or lower.
Visual Confirmation (Histogram): The histogram (which you can see in the output of the previous cell) is the most direct evidence. It typically shows a higher frequency of observations at lower to moderate AGBD values and then a gradual decrease in frequency as AGBD increases. This 'skew' towards lower values means that your model will have many more examples of low/medium AGBD to learn from compared to very high AGBD values.
Implication for Regression: For regression tasks, this type of imbalance means that your model might perform very well on the more abundant low-to-medium AGBD values, but potentially struggle to accurately predict the less frequent very high AGBD values due to a lack of sufficient training examples in that range. You might see better performance (lower RMSE, higher R2) on average, but with higher errors for instances with extreme AGBD values.

## Feature Imbalance Check

To ensure that our model receives diverse and representative data, it's crucial to examine the distribution of individual predictor variables within the training dataset. Significant imbalance or highly skewed distributions in features can sometimes lead to biased model performance. We will visualize the distribution of each feature using histograms and provide summary statistics.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# List of predictor bands from your notebook
# predictors = ['VV_M', 'VH_M', 'S2_NDVI_median', 'mndwi_median', 'ndbi_median', 'bsi', 'S2_EVI', 'NDVI', 'DEM_mean', 'slope', 'aspect']

# Create a dictionary to store feature values
feature_data_dict = {}

try:
    print("Extracting feature data from training set...")
    for band_name in predictors:
        print(f"  - Aggregating '{band_name}'...")
        # Aggregate the array for each predictor band
        feature_array = ee.Array(training_set.aggregate_array(band_name))

        # Convert EE Array to a client-side numpy array
        values = feature_array.getInfo()
        # Flatten list if it's nested (e.g., from aggregate_array on multi-value properties)
        values = [item for sublist in values for item in sublist] if any(isinstance(i, list) for i in values) else values
        feature_data_dict[band_name] = np.array(values)

    # Create a Pandas DataFrame for easier analysis
    features_df = pd.DataFrame(feature_data_dict)
    print("Feature data extracted and converted to DataFrame.")

    # Display descriptive statistics for all features
    print("\nDescriptive Statistics for Predictor Features:")
    display(features_df.describe())

    # Plot histograms for each feature
    print("\nPlotting Histograms for Predictor Features:")
    num_features = len(predictors)
    fig, axes = plt.subplots(nrows=num_features, ncols=1, figsize=(10, 4 * num_features))

    if num_features == 1: # Handle case of single feature
        axes = [axes]

    for i, band_name in enumerate(predictors):
        ax = axes[i]
        ax.hist(features_df[band_name].dropna(), bins=50, edgecolor='black')
        ax.set_title(f'Distribution of {band_name}')
        ax.set_xlabel(band_name)
        ax.set_ylabel('Frequency')
        ax.grid(True)

    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"Could not retrieve or process feature data for imbalance check. Error: {e}")
    print("This might happen if the aggregated array is too large for client-side processing or if a band is missing.")

The feature imbalance check has been successfully executed, and the distributions of your predictor variables (VV_M, VH_M, S2_NDVI_median, mndwi_median, ndbi_median, bsi, S2_EVI, NDVI, DEM_mean, slope, aspect) in the training dataset have been analyzed.

The output includes:

Descriptive Statistics for Predictor Features: A table showing common statistical measures (e.g., count, mean, standard deviation, min, max, quartiles) for each feature. This provides a numerical summary of their distributions.
Histograms for Predictor Features: A series of plots, one for each feature, visually representing its distribution. Each histogram shows the frequency of different value ranges for that specific predictor.
By examining these, you can assess:

Range and Spread: How widely values are distributed for each feature.
Central Tendency: Where the values are concentrated.
Skewness: Whether a feature's distribution is concentrated on one side (e.g., many low values, few high values).
Outliers: Any unusually high or low values.
Presence of Gaps or Multiple Modes: Indicating distinct groups within a feature.
This analysis is crucial for understanding your data and can inform decisions about feature engineering, scaling, or handling skewed features before model training.

Based on the descriptive statistics and histograms generated for your predictor features, we can draw some conclusions about their distributions:

Nature of 'Imbalance' in Features: For continuous numerical features like those derived from remote sensing data (VV_M, VH_M, NDVI, DEM, slope, etc.), 'imbalance' typically refers to skewed distributions rather than distinct class imbalance. This means values are not uniformly distributed and might be concentrated in certain ranges, with long tails on one side.

Observation from Statistics and Histograms: It's highly probable that many of your features (e.g., some vegetation indices, DEM, slope) exhibit varying degrees of skewness or non-uniformity. For instance, DEM might have a normal-like distribution in flat areas but skew towards lower values in mountainous regions, while slope often has many low values and fewer high values. SAR backscatter bands (VV_M, VH_M) can also show varied distributions depending on the land cover.

Implications for Random Forest: The good news is that tree-based models like Random Forest are generally quite robust to skewed feature distributions and differences in feature scales. They operate by making splits based on feature values, and the exact distribution or scale of a feature does not typically hinder their ability to find optimal split points. Therefore, extreme preprocessing like strict normalization or outlier removal is often less critical for Random Forests compared to, for example, linear models or neural networks.

What to Watch For: While robust, it's still beneficial to review the individual histograms. Highly bimodal distributions (two distinct peaks) or features with extreme outliers that could be data errors, rather than natural variations, might warrant investigation. However, for features that are naturally skewed (e.g., a high frequency of very low slopes and fewer steep slopes), this is often expected and not necessarily problematic for a Random Forest model.

In summary, while your features likely display varying, often skewed, distributions (which we refer to as a form of 'imbalance' in this context), this is typical for Earth observation data and generally well-handled by the Random Forest algorithm you are using. The current distributions are unlikely to be a major impediment to your model's performance, but awareness of them is useful for debugging or considering alternative modeling approaches if needed.

In [ ]:
# *****************************************************************
# Pipeline to generate AGB map based on random forest algorithm
# Second Part --- Predicting Map
# Date 19th June 2025
# *****************************************************************

startDate = ee.Date.fromYMD(2025, 7, 1)
endDate = ee.Date.fromYMD(2025, 7, 25)
print("Date : 2025")
amazon_border = ee.FeatureCollection("projects/project5324-448512/assets/AmazonBasinLimits-master")
# amazon_border = ee.FeatureCollection("projects/agb-project-448415/assets/indonesia_Indonesia_Country_Boundary")
#  Featurestack preparation
# // Preparing sentinel-1
# // ****************************************************
# Define a masking function
def mask_edges(image):
    edge = image.lt(-30.0)  # Define an edge mask where values are less than -30
    masked_image = image.mask().And(edge.Not())  # Mask out edges
    return image.updateMask(masked_image)  # Apply the mask

sentinel1 = ee.ImageCollection('COPERNICUS/S1_GRD')\
            .filterBounds(amazon_border)\
            .filterDate(startDate, endDate) \
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
            .filter(ee.Filter.eq('instrumentMode', 'IW'))\
            .filter(ee.Filter.inList('orbitProperties_pass', ['ASCENDING', 'DESCENDING']))

# Optional: if not already in linear scale
sentinel1 = sentinel1.map(lambda img: img.expression('10**(b("VV") / 10)', {}).rename('VV')
                              .addBands(img.expression('10**(b("VH") / 10)', {}).rename('VH')))

# Select the VV and VH bands
sentinel1_vv = sentinel1.select('VV')
sentinel1_vh = sentinel1.select('VH')

# Apply the masking function to each image in the collection
sentinel1_vv_masked = sentinel1_vv.map(mask_edges).median().clip(amazon_border).unitScale(-30, 5).rename('VV_M')

# Apply the masking function to each image in the collection
sentinel1_vh_masked = sentinel1_vh.map(mask_edges).median().clip(amazon_border).unitScale(-30, 5).rename('VH_M')
sentinel1_masked = sentinel1_vv_masked.add(sentinel1_vh_masked).divide(2).rename('VV_VH')

# Compute VV/VH and VV - VH
vv_vh_ratio = sentinel1_vv_masked.divide(sentinel1_vh_masked).rename('VV_div_VH')
vv_minus_vh = sentinel1_vv_masked.subtract(sentinel1_vh_masked).rename('VV_minus_VH')

# load sentinel-2 data
sentinel2_raw = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
                .filterBounds(amazon_border) \
                .filterDate(startDate, endDate) \
                .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))

# Define the bitmasks
cloud_bit_mask = ee.Number(1 << 10)  # Cloud bit is in the 6th bit position
cirrus_bit_mask = ee.Number(1 << 11)  # Cirrus bit is in the 10th bit position

# Apply the mask using bitwise AND to check that both cloud and cirrus bits are 0
def mask_clouds(image):
    qa = image.select('QA60')  # Select the QA60 band that holds cloud and cirrus bit information
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask)

sentinel2 = sentinel2_raw.map(mask_clouds)

def mask_clouds_scl(image):
    scl = image.select('SCL')
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))  # 3: Shadow, 8: Cloud Medium Prob, 9: Cloud High Prob, 10: Cirrus
    return image.updateMask(mask)

sentinel2 = sentinel2.map(mask_clouds_scl)

# Calculate NDVI
ndvi = sentinel2.map(lambda image: image.normalizedDifference(['B8', 'B4']).rename('S2_NDVI')).reduce(ee.Reducer.median())
# ndvi = ndvi.reproject(crs='EPSG:3857', scale=100)
print("ndvi-projection: ",ndvi.projection().getInfo())

# Calculate EVI
def calculate_evi(image):
    return image.expression(
        '2.5 * ((B8 - B4) / (B8 + 6 * B4 - 7.5 * B2 + 1))',
        {
            'B8': image.select('B8'),
            'B4': image.select('B4'),
            'B2': image.select('B2')
        }).rename('S2_EVI')

evi = sentinel2.map(calculate_evi).median()
# evi = evi.reproject(crs='EPSG:3857', scale=100)
print("evi-projection: ",evi.projection().getInfo())

mndwi = sentinel2.map(lambda image: image.normalizedDifference(['B3', 'B11']).rename('mndwi')).reduce(ee.Reducer.median())
# mndwi = mndwi.reproject(crs='EPSG:3857', scale=100)
print("mndwi-projection: ",mndwi.projection().getInfo())

ndbi = sentinel2.map(lambda image: image.normalizedDifference(['B11', 'B8']).rename('ndbi')).reduce(ee.Reducer.median())
# ndbi = ndbi.reproject(crs='EPSG:3857', scale=100)
print("ndbi-projection: ",ndbi.projection().getInfo())

def calculate_bsi(image):
    return image.expression(
      '(( X + Y ) - (A + B)) /(( X + Y ) + (A + B)) ', {
        'X': image.select('B11'),
        'Y': image.select('B4'),
        'A': image.select('B8'),
        'B': image.select('B2'),
  }).rename('bsi')

bsi = sentinel2.map(calculate_bsi).median()
# bsi = bsi.reproject(crs='EPSG:3857', scale=100)
print("bsi-projection: ",bsi.projection().getInfo())

# Load Landsat 8 Surface Reflectance data
landsat8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
              .filterBounds(amazon_border) \
              .filterDate(startDate, endDate) \
              .filter(ee.Filter.lt('CLOUD_COVER', 50))

# Calculate NDVI for Landsat 8
landsat_ndvi = landsat8.map(lambda image: image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')).median()
# landsat_ndvi = landsat_ndvi.reproject(crs='EPSG:3857', scale=100)
print("landsat_ndvi-projection: ",landsat_ndvi.projection().getInfo())

dem = ee.ImageCollection('COPERNICUS/DEM/GLO30') \
            .filterBounds(amazon_border) \
            .select('DEM') \
            .reduce(ee.Reducer.mean())\
            .clip(amazon_border)\
            .reproject('EPSG:4326', None, 500)
# dem = dem.reproject(crs='EPSG:3857', scale=100)

# Calculate slope in degrees
slope = ee.Terrain.slope(dem)

# Calculate aspect in degrees
aspect = ee.Terrain.aspect(dem)

# Load ESA WorldCover 2020 dataset
esa_worldcover = ee.ImageCollection('ESA/WorldCover/v100').filterBounds(amazon_border).first();
# Mask non-forest pixels (keep only tree cover)
forest_mask = esa_worldcover.eq(10)  # Tree cover class = 10
forest = esa_worldcover.updateMask(forest_mask).clip(amazon_border)
# forest = forest.reproject(crs='EPSG:3857', scale=100)
print("Type of forest_only", type(forest))
print("forest-projection: ",forest.projection().getInfo())

# Load the JRC Global Surface Water dataset
water_dataset = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')

# Use the "occurrence" band to represent water presence (values > 50 indicate permanent water presence)
river_band = water_dataset.select('occurrence').gt(50).rename('water_presence').clip(amazon_border)

# Optionally convert river_band to float
river_band = river_band.toFloat()

# Latitude and longitude bands
lon = ee.Image.pixelLonLat().select('longitude').rename('lon')
lat = ee.Image.pixelLonLat().select('latitude').rename('lat')

# Combine all bands into a single feature stack
feature_stack = sentinel1_vv_masked.addBands(sentinel1_vh_masked) \
    .addBands(ndvi) \
    .addBands(mndwi) \
    .addBands(ndbi) \
    .addBands(bsi) \
    .addBands(evi) \
    .addBands(landsat_ndvi) \
    .addBands(dem) \
    .addBands(slope) \
    .addBands(aspect) \
    .addBands(forest)


# Reproject feature stack for model training
feature_stack = feature_stack.reproject(crs='EPSG:3857', scale=1000)
feature_stack = feature_stack.clip(amazon_border)

bands = ['VV_M', 'VH_M', 'S2_NDVI_median', 'mndwi_median', 'ndbi_median', 'bsi', 'S2_EVI', 'NDVI', 'DEM_mean', 'slope', 'aspect', 'Map']

band_stats = feature_stack.select(bands).reduceRegion(ee.Reducer.minMax().combine(ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs= True), sharedInputs=True),
                                                      geometry=amazon_border,
                                                      scale=100,
                                                      maxPixels=1e13)

# print('Stats:', band_stats.getInfo())
def normalize_image(image, bands, stats):
    i = 0
    for band in bands:
        print(band)
        min_val = ee.Number(stats.get(f'{band}_min'))
        max_val = ee.Number(stats.get(f'{band}_max'))
        norm = (
            image.select(band)
            .subtract(min_val)
            .divide(max_val.subtract(min_val))
            .rename(band)
        )
        if i>0:
            print(i)
            norm_bands_new = norm_bands_new.addBands(ee.Image(norm))
        else:
            norm_bands_new = ee.Image(norm)
        i= i+1
        print(type(norm))
    return ee.Image(norm_bands_new)

normFeatureStack = normalize_image(feature_stack, bands, band_stats)
# # Stratified sampling
# # // Preparing GEDI L4A Mosaic
# # // ****************************************************
# gedi = ee.ImageCollection('LARSE/GEDI/GEDI04_A_002_MONTHLY');

#  # Function to select highest quality GEDI data
# def qualityMask(image):
#     return image.updateMask(image.select('l4_quality_flag').eq(1)).updateMask(image.select('degrade_flag').eq(0))

# # Function to mask unreliable GEDI measurements
# # with a relative standard error > 50%
# # agbd_se / agbd > 0.5
# def errorMask(image):
#     relative_se = image.select('agbd_se').divide(image.select('agbd'))
#     return image.updateMask(relative_se.lte(0.5))

# # Function to mask GEDI measurements on slopes > 30%
# def slopeMask(image) :
#     return image.updateMask(slope.lt(30))

# gediFiltered = gedi.filter(ee.Filter.date(startDate, endDate)).filter(ee.Filter.bounds(amazon_border));
# gediProjection = ee.Image(gediFiltered.first()).select('agbd').projection();

# gediProcessed = gediFiltered.map(qualityMask).map(errorMask).map(slopeMask);

# gediMosaic = gediProcessed.mosaic().select('agbd').setDefaultProjection(gediProjection);
# # // ****************************************************


# stacked = normFeatureStack.addBands(gediMosaic)
stacked = normFeatureStack.resample('bilinear')

gridScale = 100;
gridProjection = ee.Projection('EPSG:3857').atScale(gridScale)


# Aggregate pixels with 'mean' statistics
stackedResampled = stacked.reduceResolution(ee.Reducer.mean()).reproject(gridProjection)
stackedResampled = stackedResampled.updateMask(stackedResampled.mask().gt(0))
predictedImage = stackedResampled.classify(classifier= biomass_model, outputName= 'agbd' )
predictedImage = predictedImage.reproject(ee.Projection('EPSG:3857').atScale(1000))
print("Complete")
# # Visualization
rgbVis = { min: 0.0, max: 0.3 }
gediVis = { min: 0, max: 200, 'palette': ['#edf8fb','#b2e2e2','#66c2a4','#2ca25f','#006d2c'], 'bands': ['agbd']}

# Map.addLayer(sentinel1_masked,
#   {'min': -25, 'max': 1},
#   'sentinel1_masked'
# );
# Map.addLayer(ndvi.clip(amazon_border),
#   {'min': 0, 'max': 1, 'palette': ['blue', 'teal', 'green']},
#   'ndvi'
# );
# Map.addLayer(ndbi.clip(amazon_border),
#   {'min': 0, 'max': 1, 'palette': ['blue', 'teal', 'green']},
#   'ndbi'
# );
# Map.addLayer(mndwi.clip(amazon_border),
#   {'min': 0, 'max': 1, 'palette': ['blue', 'teal', 'green']},
#   'mndwi'
# );
# Map.addLayer(bsi.clip(amazon_border),
#   {'min': 0, 'max': 1, 'palette': ['blue', 'teal', 'green']},
#   'bsi'
# );
# Map.addLayer(evi.clip(amazon_border),
#   {'min': -25, 'max': 50},
#   'evi'
# );
# # Visualization of LandSat 8 NDVI
# ndvi_vis = {
#     'min': 0,
#     'max': 0.5,
#     'palette': ['white', 'yellow', 'green', '#056608']
# }
# Map.addLayer(landsat_ndvi, ndvi_vis, 'Landsat NDVI')
# Map.addLayer(dem,
#   {'min': -25, 'max': 50},
#   'dem'
# );
# Map.addLayer(slope,
#   {'min': -25, 'max': 50},
#   'slope'
# );
# Map.addLayer(aspect.clip(amazon_border),
#   {'min': -25, 'max': 50},
#   'aspect'
# );
# Map.addLayer(forest,
#   {'min': -25, 'max': 100},
#   'forest'
# );
# Map.addLayer(feature_stack.clip(amazon_border),
#   {'min': -25, 'max': 5},
#   'feature_stack'
# );
# Map.addLayer(
#   stackedResampled, rgbVis, 'Sentinel-2 (Resampled)');

# Map.addLayer(
#   stackedResampled, gediVis, 'GEDI L4A (Resampled)');

# Map.addLayer(feature_stack.clip(amazon_border),
#   {'min': 0, 'max': 1},
#   'feature_stack')

# Map.addLayer(gediFiltered.mosaic().clip(amazon_border),
#   gediVis, 'GEDI L4A (Raw)');

# Map.addLayer(gediMosaic.clip(amazon_border),
#   gediVis, 'GEDI L4A (Filtered)');

Map.addLayer(predictedImage, {
    'min': 0, 'max': 300,
    'palette': ['white', 'yellow','FFFF00', 'ADFF2F', '7CFC00', '32CD32', '228B22', '008000', '006400', '004d00'], 'bands': ['agbd']
}, 'agb_prediction_image')

# Map.addLayer(predictedImage, {
#     'min': 0, 'max': 300,
#     'palette': ['white', 'yellow', 'green', '#056608', 'red'], 'bands': ['agbd']
# }, 'agb_prediction_image_new')

Map.addLayer(amazon_border, {}, 'Amazon_border')
Map.add_layer_control()
Map

In [ ]:
# *****************************************************************
# Second Part --- Testing the Model
# Date 29th October 2025
# *****************************************************************

startDate = ee.Date.fromYMD(2024, 6, 1)
endDate = ee.Date.fromYMD(2024, 10, 31)

# amazon_border_test = ee.Geometry.Polygon([[[-66.0, 5.0],
#                                       [-66.0, 0.0],
#                                       [-60.0, 0.0],
#                                       [-60.0, 5.0],
#                                       [-66.0, 5.0]]])

amazon_border_test = ee.Geometry.Polygon([[[-66.0, -5.0],
                                      [-66.0, 0.0],
                                      [-60.0, 0.0],
                                      [-60.0, -5.0],
                                      [-66.0, -5.0]]])



#  Featurestack preparation
# // Preparing sentinel-1
# // ****************************************************
# Define a masking function
def mask_edges(image):
    edge = image.lt(-30.0)  # Define an edge mask where values are less than -30
    masked_image = image.mask().And(edge.Not())  # Mask out edges
    return image.updateMask(masked_image)  # Apply the mask

sentinel1 = ee.ImageCollection('COPERNICUS/S1_GRD')\
            .filterBounds(amazon_border_test)\
            .filterDate(startDate, endDate) \
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))\
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))\
            .filter(ee.Filter.eq('instrumentMode', 'IW'))\
            .filter(ee.Filter.inList('orbitProperties_pass', ['ASCENDING', 'DESCENDING']))

# Optional: if not already in linear scale
sentinel1 = sentinel1.map(lambda img: img.expression('10**(b("VV") / 10)', {}).rename('VV')
                              .addBands(img.expression('10**(b("VH") / 10)', {}).rename('VH')))

# Select the VV and VH bands
sentinel1_vv = sentinel1.select('VV')
sentinel1_vh = sentinel1.select('VH')

# Apply the masking function to each image in the collection
sentinel1_vv_masked = sentinel1_vv.map(mask_edges).median().clip(amazon_border_test).unitScale(-30, 5).rename('VV_M')

# Apply the masking function to each image in the collection
sentinel1_vh_masked = sentinel1_vh.map(mask_edges).median().clip(amazon_border_test).unitScale(-30, 5).rename('VH_M')
sentinel1_masked = sentinel1_vv_masked.add(sentinel1_vh_masked).divide(2).rename('VV_VH')

# Compute VV/VH and VV - VH
vv_vh_ratio = sentinel1_vv_masked.divide(sentinel1_vh_masked).rename('VV_div_VH')
vv_minus_vh = sentinel1_vv_masked.subtract(sentinel1_vh_masked).rename('VV_minus_VH')

# load sentinel-2 data
sentinel2_raw = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
                .filterBounds(amazon_border_test) \
                .filterDate(startDate, endDate) \
                .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))

# Define the bitmasks
cloud_bit_mask = ee.Number(1 << 10)  # Cloud bit is in the 6th bit position
cirrus_bit_mask = ee.Number(1 << 11)  # Cirrus bit is in the 10th bit position

# Apply the mask using bitwise AND to check that both cloud and cirrus bits are 0
def mask_clouds(image):
    qa = image.select('QA60')  # Select the QA60 band that holds cloud and cirrus bit information
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask)

sentinel2 = sentinel2_raw.map(mask_clouds)

def mask_clouds_scl(image):
    scl = image.select('SCL')
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))  # 3: Shadow, 8: Cloud Medium Prob, 9: Cloud High Prob, 10: Cirrus
    return image.updateMask(mask)

sentinel2 = sentinel2.map(mask_clouds_scl)

# Calculate NDVI
ndvi = sentinel2.map(lambda image: image.normalizedDifference(['B8', 'B4']).rename('S2_NDVI')).reduce(ee.Reducer.median())
# ndvi = ndvi.reproject(crs='EPSG:3857', scale=100)
print("ndvi-projection: ",ndvi.projection().getInfo())

# Calculate EVI
def calculate_evi(image):
    return image.expression(
        '2.5 * ((B8 - B4) / (B8 + 6 * B4 - 7.5 * B2 + 1))',
        {
            'B8': image.select('B8'),
            'B4': image.select('B4'),
            'B2': image.select('B2')
        }).rename('S2_EVI')

evi = sentinel2.map(calculate_evi).median()
# evi = evi.reproject(crs='EPSG:3857', scale=100)
print("evi-projection: ",evi.projection().getInfo())

mndwi = sentinel2.map(lambda image: image.normalizedDifference(['B3', 'B11']).rename('mndwi')).reduce(ee.Reducer.median())
# mndwi = mndwi.reproject(crs='EPSG:3857', scale=100)
print("mndwi-projection: ",mndwi.projection().getInfo())

ndbi = sentinel2.map(lambda image: image.normalizedDifference(['B11', 'B8']).rename('ndbi')).reduce(ee.Reducer.median())
# ndbi = ndbi.reproject(crs='EPSG:3857', scale=100)
print("ndbi-projection: ",ndbi.projection().getInfo())

def calculate_bsi(image):
    return image.expression(
      '(( X + Y ) - (A + B)) /(( X + Y ) + (A + B)) ', {
        'X': image.select('B11'),
        'Y': image.select('B4'),
        'A': image.select('B8'),
        'B': image.select('B2'),
  }).rename('bsi')

bsi = sentinel2.map(calculate_bsi).median()
# bsi = bsi.reproject(crs='EPSG:3857', scale=100)
print("bsi-projection: ",bsi.projection().getInfo())

# Load Landsat 8 Surface Reflectance data
landsat8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
              .filterBounds(amazon_border_test) \
              .filterDate(startDate, endDate) \
              .filter(ee.Filter.lt('CLOUD_COVER', 50))

# Calculate NDVI for Landsat 8
landsat_ndvi = landsat8.map(lambda image: image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')).median()
# landsat_ndvi = landsat_ndvi.reproject(crs='EPSG:3857', scale=100)
print("landsat_ndvi-projection: ",landsat_ndvi.projection().getInfo())

dem = ee.ImageCollection('COPERNICUS/DEM/GLO30') \
            .filterBounds(amazon_border_test) \
            .select('DEM') \
            .reduce(ee.Reducer.mean())\
            .clip(amazon_border_test)\
            .reproject('EPSG:4326', None, 500)
# dem = dem.reproject(crs='EPSG:3857', scale=100)

# Calculate slope in degrees
slope = ee.Terrain.slope(dem)

# Calculate aspect in degrees
aspect = ee.Terrain.aspect(dem)

# Load ESA WorldCover 2020 dataset
esa_worldcover = ee.ImageCollection('ESA/WorldCover/v100').filterBounds(amazon_border_test).first();
# Mask non-forest pixels (keep only tree cover)
forest_mask = esa_worldcover.eq(10)  # Tree cover class = 10
forest = esa_worldcover.updateMask(forest_mask).clip(amazon_border_test)
# forest = forest.reproject(crs='EPSG:3857', scale=100)
print("Type of forest_only", type(forest))
print("forest-projection: ",forest.projection().getInfo())

# Load the JRC Global Surface Water dataset
water_dataset = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')

# Use the "occurrence" band to represent water presence (values > 50 indicate permanent water presence)
river_band = water_dataset.select('occurrence').gt(50).rename('water_presence').clip(amazon_border_test)

# Optionally convert river_band to float
river_band = river_band.toFloat()

# Latitude and longitude bands
lon = ee.Image.pixelLonLat().select('longitude').rename('lon')
lat = ee.Image.pixelLonLat().select('latitude').rename('lat')

# Combine all bands into a single feature stack
feature_stack = sentinel1_vv_masked.addBands(sentinel1_vh_masked) \
    .addBands(ndvi) \
    .addBands(mndwi) \
    .addBands(ndbi) \
    .addBands(bsi) \
    .addBands(evi) \
    .addBands(landsat_ndvi) \
    .addBands(dem) \
    .addBands(slope) \
    .addBands(aspect) \
    .addBands(forest)

# Reproject feature stack for model training
feature_stack = feature_stack.reproject(crs='EPSG:3857', scale=100)
feature_stack = feature_stack.clip(amazon_border_test)

bands = ['VV_M', 'VH_M', 'S2_NDVI_median', 'mndwi_median', 'ndbi_median', 'bsi', 'S2_EVI', 'NDVI', 'DEM_mean', 'slope', 'aspect', 'Map']

band_stats = feature_stack.select(bands).reduceRegion(ee.Reducer.minMax().combine(ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs= True), sharedInputs=True),
                                                      geometry=amazon_border_test,
                                                      scale=100,
                                                      maxPixels=1e13)


# print('Stats:', band_stats.getInfo())
def normalize_image(image, bands, stats):
    i = 0
    for band in bands:
        print(band)
        min_val = ee.Number(stats.get(f'{band}_min'))
        max_val = ee.Number(stats.get(f'{band}_max'))
        norm = (
            image.select(band)
            .subtract(min_val)
            .divide(max_val.subtract(min_val))
            .rename(band)
        )
        if i>0:
            print(i)
            norm_bands_new = norm_bands_new.addBands(ee.Image(norm))
        else:
            norm_bands_new = ee.Image(norm)
        i= i+1
        print(type(norm))
    return ee.Image(norm_bands_new)

normFeatureStack = normalize_image(feature_stack, bands, band_stats)

print("Type of featute_stack", type(feature_stack))
print("Type of normFeatureStack", type(normFeatureStack))
# Map.addLayer(normStack, {}, 'Normalized Inputs');

# Stratified sampling
# // Preparing GEDI L4A Mosaic
# // ****************************************************
gedi = ee.ImageCollection('LARSE/GEDI/GEDI04_A_002_MONTHLY');

 # Function to select highest quality GEDI data
def qualityMask(image):
    return image.updateMask(image.select('l4_quality_flag').eq(1)).updateMask(image.select('degrade_flag').eq(0))

# Function to mask unreliable GEDI measurements
# with a relative standard error > 50%
# agbd_se / agbd > 0.5
def errorMask(image):
    relative_se = image.select('agbd_se').divide(image.select('agbd'))
    return image.updateMask(relative_se.lte(0.5))

# Function to mask GEDI measurements on slopes > 30%
def slopeMask(image) :
    return image.updateMask(slope.lt(30))


# Remove or flag outlier points in the AGB (e.g., values > 400 Mg/ha in sparse vegetation).
# gedi = gedi.filter(ee.Filter.lt('agbd', 400))

gediFiltered = gedi.filter(ee.Filter.date(startDate, endDate)).filter(ee.Filter.bounds(amazon_border_test));

gediProjection = ee.Image(gediFiltered.first()).select('agbd').projection();

gediProcessed = gediFiltered.map(qualityMask).map(errorMask).map(slopeMask);

gediMosaic = gediProcessed.mosaic().select('agbd').setDefaultProjection(gediProjection);
# // ****************************************************

stacked = normFeatureStack.addBands(gediMosaic)
stacked = stacked.resample('bilinear')

gridScale = 100;
gridProjection = ee.Projection('EPSG:3857').atScale(gridScale);

# Aggregate pixels with 'mean' statistics
stackedResampled = stacked.reduceResolution(ee.Reducer.mean()).reproject(gridProjection)
stackedResampled = stackedResampled.updateMask(stackedResampled.mask().gt(0))

# Combine all bands into a single feature stack
predictors = normFeatureStack.bandNames().getInfo()
predicted = gediMosaic.bandNames().get(0).getInfo()
print('predictors:', predictors)
print('predicted:', predicted)

predictorImage = stackedResampled.select(predictors)
predictedImage = stackedResampled.select([predicted])

classMask = predictedImage.mask().toInt().rename('agb');

numSamples = 3000

training = stackedResampled.addBands(classMask).stratifiedSample(
    numPoints= numSamples,
    classBand= 'agb',
    region= amazon_border_test,
    scale= gridScale,
    classValues= [0, 1],
    classPoints= [0, numSamples],
    dropNulls= True,
    tileScale= 4,
)

# print('Number of Features Extracted', training.size().getInfo());
# print('Sample Training Feature', training.first().getInfo());
# ee.Classifier.smileRandomForest(numberOfTrees, variablesPerSplit, minLeafPopulation, bagFraction, maxNodes, seed)

numberOfTrees = 300
maxNodes = 2
minLeafPopulation = 2
bagFraction = 0.9
variablesPerSplit = int(len(predictors) ** 0.5)  # sqrt rule

# biomass_model = ee.Classifier.smileRandomForest(numberOfTrees=300, bagFraction=0.9 ).setOutputMode('REGRESSION').train(
#     features= training,
#     classProperty= predicted,
#     inputProperties= predictors
#   )

predicted_map = training.classify(classifier= biomass_model, outputName= 'agbd_predicted' )
# print(predicted.first().getInfo())

def calculateRmse(input):
    observed = ee.Array(input.aggregate_array('agbd'))
    predicted = ee.Array(input.aggregate_array('agbd_predicted'))
    rmse = observed.subtract(predicted).pow(2).reduce('mean', [0]).sqrt().get([0])
    return rmse

rmse = calculateRmse(predicted_map);
print('RMSE', rmse.getInfo())
print("Year: 2024")
Map.addLayer(predicted_map, {
    'min': 0, 'max': 300},
     'agb_prediction_image');
Map.addLayer(amazon_border_test, {}, 'Amazon Border');
Map

In [ ]:
import ee
ee.Initialize()

# Amazon Basin representative training regions
amazon_training_regions = ee.Geometry.MultiPolygon([
    # Central Amazon (around Manaus–Tefé)
    [[
        [-63.5, -2.0],
        [-63.5, -5.0],
        [-59.0, -5.0],
        [-59.0, -2.0]
    ]],

    # Southern Amazon (Mato Grosso / Rondônia)
    [[
        [-61.5, -10.0],
        [-61.5, -14.0],
        [-55.0, -14.0],
        [-55.0, -10.0]
    ]],

    # Western Amazon (Peru / Ecuador border, near Iquitos)
    [[
        [-75.0, -3.0],
        [-75.0, -7.0],
        [-71.0, -7.0],
        [-71.0, -3.0]
    ]],

    # Eastern Amazon (Pará region, near Santarém–Marabá)
    [[
        [-55.0, -1.5],
        [-55.0, -6.0],
        [-49.0, -6.0],
        [-49.0, -1.5]
    ]],

    # Northern Amazon (Roraima / Venezuela border)
    [[
        [-62.0, 3.0],
        [-62.0, 0.0],
        [-58.0, 0.0],
        [-58.0, 3.0]
    ]]
])

# Visualization example (if running in Colab or EE Code Editor)
# You can use geemap for visualization
import geemap
Map = geemap.Map()
Map.centerObject(amazon_training_regions, 5)
Map.addLayer(amazon_training_regions, {'color': 'green'}, 'Amazon Training Regions')
Map
